# Analisis Sentimen E-Commerce
### Dataset: `dataset_sentimen_ecommerce_3.csv`

**Tujuan:** Mengolah data review e-commerce, menambahkan label sentimen berdasarkan rating dan analisis lexicon Bahasa Indonesia, lalu mengekspor hasil siap digunakan di Power BI.

---

**Arsitektur analisis (sesuai modul PDF):**
```
Data Teks
    ↓
Preprocessing
    ↓
Analisis Sentimen (Rating + Lexicon Indonesia)
    ↓
Hasil skor sentimen
    ↓
Visualisasi di Power BI
```

## Import Library

In [10]:
import pandas as pd
import re

print('Library berhasil diimport')
print(f'   pandas versi: {pd.__version__}')

Library berhasil diimport
   pandas versi: 3.0.1


## Load Dataset

In [11]:
# Sesuaikan path jika file berada di folder lain
FILE_PATH = 'dataset_sentimen_ecommerce_3.csv'

df = pd.read_csv(FILE_PATH, sep=';')

print(f'Dataset berhasil dimuat')
print(f'   Jumlah baris : {len(df)}')
print(f'   Jumlah kolom : {len(df.columns)}')
print(f'   Nama kolom   : {df.columns.tolist()}')
print()
df.head(10)

Dataset berhasil dimuat
   Jumlah baris : 100
   Jumlah kolom : 5
   Nama kolom   : ['id_review', 'marketplace', 'produk', 'rating', 'komentar']



,id_review,marketplace,produk,rating,komentar
0,1,Blibli,Power Bank,1,Barang rusak saat diterima
1,2,Lazada,Keyboard,4,Belanja di sini memuaskan
2,3,Blibli,Headset,1,Produk tidak sesuai gambar
3,4,Tokopedia,Power Bank,4,Produk sangat bagus dan sesuai deskripsi
4,5,Blibli,Keyboard,2,Barang rusak saat diterima
5,6,Blibli,Jam Tangan,1,Produk cepat rusak
6,7,Tokopedia,Smartphone,2,Barang palsu dan tidak original
7,8,Tokopedia,Headset,2,Produk cepat rusak
8,9,Blibli,Power Bank,3,Barang sesuai ukuran
9,10,Tokopedia,Laptop,1,Produk tidak sesuai gambar


## Eksplorasi Data Awal (EDA)

In [12]:
print('=' * 45)
print(' INFO TIPE DATA')
print('=' * 45)
print(df.dtypes)
print()

print('=' * 45)
print(' MISSING VALUE')
print('=' * 45)
print(df.isnull().sum())
print()

print('=' * 45)
print(' DISTRIBUSI RATING')
print('=' * 45)
print(df['rating'].value_counts().sort_index())
print()

print('=' * 45)
print(' JUMLAH REVIEW PER MARKETPLACE')
print('=' * 45)
print(df['marketplace'].value_counts())
print()

print('=' * 45)
print(' JUMLAH REVIEW PER PRODUK')
print('=' * 45)
print(df['produk'].value_counts())

 INFO TIPE DATA
id_review      int64
marketplace      str
produk           str
rating         int64
komentar         str
dtype: object

 MISSING VALUE
id_review      0
marketplace    0
produk         0
rating         0
komentar       0
dtype: int64

 DISTRIBUSI RATING
rating
1    18
2    18
3    34
4    19
5    11
Name: count, dtype: int64

 JUMLAH REVIEW PER MARKETPLACE
marketplace
Tokopedia    29
Lazada       27
Shopee       25
Blibli       19
Name: count, dtype: int64

 JUMLAH REVIEW PER PRODUK
produk
Baju          16
Keyboard      14
Smartphone    14
Headset       11
Power Bank    10
Mouse         10
Sepatu         8
Jam Tangan     7
Laptop         6
Tas            4
Name: count, dtype: int64


## Preprocessing Teks Komentar



In [13]:
def bersihkan_teks(teks):
    """
    Preprocessing teks komentar:
    1. Ubah ke huruf kecil (lowercase)
    2. Hapus karakter selain huruf dan spasi
    3. Hapus spasi berlebih
    """
    teks = str(teks).lower()                    # lowercase
    teks = re.sub(r'[^a-zA-Z\s]', ' ', teks)   # hapus angka & simbol
    teks = re.sub(r'\s+', ' ', teks).strip()    # hapus spasi ganda
    return teks

df['komentar_bersih'] = df['komentar'].apply(bersihkan_teks)

print('Preprocessing selesai. Contoh hasil:')
df[['komentar', 'komentar_bersih']].head(8)

Preprocessing selesai. Contoh hasil:


,komentar,komentar_bersih
0,Barang rusak saat diterima,barang rusak saat diterima
1,Belanja di sini memuaskan,belanja di sini memuaskan
2,Produk tidak sesuai gambar,produk tidak sesuai gambar
3,Produk sangat bagus dan sesuai deskripsi,produk sangat bagus dan sesuai deskripsi
4,Barang rusak saat diterima,barang rusak saat diterima
5,Produk cepat rusak,produk cepat rusak
6,Barang palsu dan tidak original,barang palsu dan tidak original
7,Produk cepat rusak,produk cepat rusak


## Labeling Sentimen Berdasarkan Rating

Rating **≥ 4** → **Positif**
Rating **= 3** → **Netral**
Rating **≤ 2** → **Negatif**

In [14]:
def label_sentimen(rating):
    """
    Fungsi labeling sentimen dari kolom rating.
    Sesuai script Python di modul PDF.
    """
    if rating >= 4:
        return 'Positif'
    elif rating == 3:
        return 'Netral'
    else:
        return 'Negatif'

# Membuat kolom sentimen — sesuai modul PDF
df['sentimen'] = df['rating'].apply(label_sentimen)

print('Kolom sentimen berhasil dibuat')
print()
print('Distribusi Sentimen:')
dist = df['sentimen'].value_counts()
pct  = (df['sentimen'].value_counts(normalize=True) * 100).round(1)
for label in dist.index:
    print(f'  {label:10s}: {dist[label]:3d} ({pct[label]}%)')

Kolom sentimen berhasil dibuat

Distribusi Sentimen:
  Negatif   :  36 (36.0%)
  Netral    :  34 (34.0%)
  Positif   :  30 (30.0%)


## Analisis Sentimen dengan Lexicon Bahasa Indonesia (Bonus)

In [15]:
# Kamus Lexicon Bahasa Indonesia
KATA_POSITIF = [
    'bagus', 'baik', 'puas', 'memuaskan', 'cepat', 'oke', 'mantap',
    'sempurna', 'sesuai', 'original', 'kualitas', 'recommended',
    'rekomendasi', 'suka', 'senang', 'deskripsi', 'aman', 'terpercaya',
    'murah', 'terjangkau', 'rapi', 'nyaman', 'berkualitas', 'terbaik'
]

KATA_NEGATIF = [
    'buruk', 'rusak', 'lambat', 'lama', 'jelek', 'kecewa', 'tidak',
    'palsu', 'bohong', 'tipu', 'salah', 'hancur', 'cacat', 'kurang',
    'mahal', 'mengecewakan', 'gagal', 'kotor', 'patah', 'retak'
]


def cek_sentimen_lexicon(teks):
    """
    Menghitung skor sentimen berdasarkan kemunculan
    kata positif dan negatif dalam teks komentar.
    (Referensi: Opsi C modul PDF)
    """
    teks = str(teks).lower()
    skor = 0

    for kata in KATA_POSITIF:
        if kata in teks:
            skor += 1

    for kata in KATA_NEGATIF:
        if kata in teks:
            skor -= 1

    if skor > 0:
        return 'Positif', skor
    elif skor < 0:
        return 'Negatif', skor
    else:
        return 'Netral', skor


hasil = df['komentar_bersih'].apply(cek_sentimen_lexicon)
df['sentimen_lexicon'] = hasil.apply(lambda x: x[0])
df['skor_lexicon']     = hasil.apply(lambda x: x[1])

print('Analisis Lexicon selesai')
print()
print('Distribusi Sentimen Lexicon:')
print(df['sentimen_lexicon'].value_counts())
print()
print('Contoh hasil:')
df[['komentar', 'sentimen_lexicon', 'skor_lexicon']].head(8)

Analisis Lexicon selesai

Distribusi Sentimen Lexicon:
sentimen_lexicon
Netral     41
Positif    37
Negatif    22
Name: count, dtype: int64

Contoh hasil:


,komentar,sentimen_lexicon,skor_lexicon
0,Barang rusak saat diterima,Negatif,-1
1,Belanja di sini memuaskan,Positif,1
2,Produk tidak sesuai gambar,Netral,0
3,Produk sangat bagus dan sesuai deskripsi,Positif,3
4,Barang rusak saat diterima,Negatif,-1
5,Produk cepat rusak,Netral,0
6,Barang palsu dan tidak original,Negatif,-1
7,Produk cepat rusak,Netral,0


## Preview Dataset

In [16]:
# Susun kolom sesuai urutan final
KOLOM_FINAL = [
    'id_review',
    'marketplace',
    'produk',
    'rating',
    'komentar',
    'komentar_bersih',
    'sentimen_lexicon',
    'skor_lexicon',
    'sentimen',          # kolom utama sesuai PDF
]

df_final = df[KOLOM_FINAL].copy()

print('=' * 50)
print(' DATASET FINAL')
print('=' * 50)
print(f'Jumlah baris : {len(df_final)}')
print(f'Jumlah kolom : {len(df_final.columns)}')
print()
print('Kolom:')
for i, col in enumerate(df_final.columns, 1):
    print(f'  {i}. {col}')
print()
df_final.head(10)

 DATASET FINAL
Jumlah baris : 100
Jumlah kolom : 9

Kolom:
  1. id_review
  2. marketplace
  3. produk
  4. rating
  5. komentar
  6. komentar_bersih
  7. sentimen_lexicon
  8. skor_lexicon
  9. sentimen



,id_review,marketplace,produk,rating,komentar,komentar_bersih,sentimen_lexicon,skor_lexicon,sentimen
0,1,Blibli,Power Bank,1,Barang rusak saat diterima,barang rusak saat diterima,Negatif,-1,Negatif
1,2,Lazada,Keyboard,4,Belanja di sini memuaskan,belanja di sini memuaskan,Positif,1,Positif
2,3,Blibli,Headset,1,Produk tidak sesuai gambar,produk tidak sesuai gambar,Netral,0,Negatif
3,4,Tokopedia,Power Bank,4,Produk sangat bagus dan sesuai deskripsi,produk sangat bagus dan sesuai deskripsi,Positif,3,Positif
4,5,Blibli,Keyboard,2,Barang rusak saat diterima,barang rusak saat diterima,Negatif,-1,Negatif
5,6,Blibli,Jam Tangan,1,Produk cepat rusak,produk cepat rusak,Netral,0,Negatif
6,7,Tokopedia,Smartphone,2,Barang palsu dan tidak original,barang palsu dan tidak original,Negatif,-1,Negatif
7,8,Tokopedia,Headset,2,Produk cepat rusak,produk cepat rusak,Netral,0,Negatif
8,9,Blibli,Power Bank,3,Barang sesuai ukuran,barang sesuai ukuran,Positif,1,Netral
9,10,Tokopedia,Laptop,1,Produk tidak sesuai gambar,produk tidak sesuai gambar,Netral,0,Negatif


## Export CSV

In [17]:
OUTPUT_FILE = 'hasil_sentimen_final.csv'

df_final.to_csv(OUTPUT_FILE, index=False, sep=';', encoding='utf-8-sig')
# utf-8-sig: agar karakter Indonesia tampil benar di Excel & Power BI

print(f'File berhasil disimpan: {OUTPUT_FILE}')
print(f'   Jumlah baris : {len(df_final)}')
print(f'   Jumlah kolom : {len(df_final.columns)}')
print()

File berhasil disimpan: hasil_sentimen_final.csv
   Jumlah baris : 100
   Jumlah kolom : 9

